# Speech Pretraining: quantization and losses 

This notebook explains several important components used in self‑supervised speech models such as **Wav2Vec2** and **BestRQ**.

Goals:
- Understand **vector quantization with Gumbel‑Softmax**
- Understand **contrastive learning with distractors**
- Implement several components (**BEST-RQ quantization**, **contrastive loss**) yourself as homework

## Key Idea

Self‑supervised speech models don't train with labels.
Instead:

1. Encode audio into latent representations
2. Quantize those representations into discrete tokens
3. Compute loss (contrastive loss, masked language model loss)

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Gumbel

## Wav2Vec2 Quantizer

This module converts continuous representations into **discrete codebook vectors**.

Steps:

1. Project input to logits for all codebooks
2. Add **Gumbel noise**
3. Perform differentiable **straight through Gumbel‑Softmax trick**
4. Convert one‑hot vectors to embeddings


In [ ]:
class Wav2Ve2Quantizer(nn.Module):
    def __init__(self, dim: int, vocab_size: int, codebooks: int, out_dim: int):
        super().__init__()
        

    def forward(self, x, temperature):
        

### Quick Test

In [ ]:
B, T, D = 2, 100, 256

x = torch.randn(B, T, D)

quantizer = Wav2Ve2Quantizer(
    dim=256,
    vocab_size=320,
    codebooks=2,
    out_dim=256
)

out = quantizer(x, temperature=0.5)
print(out.shape)

# Homework Section

## Contrastive Loss (2.5 points)
**Task**: implement [wav2vec2](https://arxiv.org/pdf/2006.11477)-based contrastive loss.

Steps:
- Compute cosine similarity between predictions and true targets
- Sample **distractor negatives** from other time steps
- Compute logaddexp over negatives logits and add with positive logit 


In [ ]:
class ContrastiveLoss(nn.Module):

    def __init__(self, temperature: int, distractor_count: int):
        super().__init__()
        self.temperature = temperature
        self.distractor_count = distractor_count

    def forward(self, output, quantized, masked):
        """
        Args:
            output: (B, T, D)
            quantized: (B, T, D)
            masked: (B, T)
        1. Sample distractor indices
        2. Gather distractor vectors
        3. Compute cosine similarities
        4. Compute contrastive loss
        """
        B, T, D = quantized.shape
        distractor_idxs = torch.randint(0, T, (B, T, self.distractor_count), device=quantized.device)

        # YOUR CODE HERE

        raise NotImplementedError()

## Combined W2V-BERT loss (0.5 points)
**Task**: implement combined [W2V-BERT](https://arxiv.org/abs/2108.06209) loss.

Steps:
- Contrastive prediction loss
- MLM‑style token prediction loss


In [ ]:
class Wav2Vec2BertLoss(nn.Module):

    def __init__(self, temperature: int, distractor_count: int, beta: float):
        super().__init__()

        self.beta = beta
        self.contrastive_loss = ContrastiveLoss(temperature, distractor_count)

    def forward(self, output, mlm_output, quantized, quantized_ids, masked):
        """
        Args:
            output: (B, T, D)
            mlm_output: (B, T, D)
            quantized: (B, T, D)
            quantized_ids: (B, T, D)
            masked: (B, T)
        1. Compute contrastive loss
        2. Compute MLM loss
        3. Combine them using beta
        """

        # YOUR CODE HERE

        raise NotImplementedError()

## BestRQ Quantizer (2 points)

[BestRQ](https://arxiv.org/pdf/2202.01855) quantizer uses **random projections + nearest neighbor lookup** instead of Gumbel Softmax from Wav2Vec2.

Steps:

1. Project input with a fixed random matrix
2. Normalize vectors
3. Compute similarity with codebook embeddings
4. Select the closest embedding


In [ ]:
class BestRQQuantizer(nn.Module):

    def __init__(self, dim: int, vocab_size: int, codebooks: int, hid_dim: int):
        super().__init__()

        """
        TODO:
        1. Create fixed random embeddings
        2. Normalize them
        3. Create random projection matrix
        """

        raise NotImplementedError()

    def forward(self, x):

        """
        Args:
            x: (B, T, D)
        1. Project input
        2. Normalize
        3. Compute similarity
        4. Return token indices
        """

        raise NotImplementedError()